## Setup

In [1]:
import importlib
from pathlib import Path
import sys
import polars as pl

pl.Config.set_tbl_rows(25)

REPO_DIR = Path('.').resolve().parent
DEV_DATA_DIR = REPO_DIR / 'trio_dev_data'

sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'src' / 'util'))

from version_sort import version_sort


In [2]:
KID_ID = 'NA12878'
DAD_ID = 'NA12891'
MOM_ID = 'NA12892'

METH_BASE_DIR = DEV_DATA_DIR / 'output' / 'dna-methylation'

# Unphased (combined) methylation BEDs from aligned_bam_to_cpg_scores
METH_COUNT_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.count.pedmec-phased' # output dir of aligned_bam_to_cpg_scores (containing count-based unphased meth)
METH_MODEL_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.model.pedmec-phased' # output dir of aligned_bam_to_cpg_scores (containing model-based unphased meth)

def combined_meth_paths(uid):
    """Return (count_combined, model_combined) paths for a sample."""
    return (
        str(METH_COUNT_DIR / f'{uid}.GRCh38.haplotagged.combined.bed.gz'), # unphased count-based meth from aligned_bam_to_cpg_scores
        str(METH_MODEL_DIR / f'{uid}.GRCh38.haplotagged.combined.bed.gz'), # unphased model-based meth from aligned_bam_to_cpg_scores
    )

KID_METH_COUNT_COMBINED, KID_METH_MODEL_COMBINED = combined_meth_paths(KID_ID)
DAD_METH_COUNT_COMBINED, DAD_METH_MODEL_COMBINED = combined_meth_paths(DAD_ID)
MOM_METH_COUNT_COMBINED, MOM_METH_MODEL_COMBINED = combined_meth_paths(MOM_ID)

# Parent-phased methylation BED
PARENT_PHASED_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.parent-phased' # output dir of phase_meth_to_parent_haps.py
BED_METH_PARENT_PHASED = str(PARENT_PHASED_DIR / 'trio.dna-methylation.bed') # parent-phased methylation levels from phase_meth_to_parent_haps.py

# Heterozygous sites at which bit-vectors are mismatched, from phase_meth_to_parent_haps.py
BED_HET_SITE_MISMATCHES_PAT = str(PARENT_PHASED_DIR / f'{KID_ID}.bit-vector-sites-mismatches.paternal.bed')
BED_HET_SITE_MISMATCHES_MAT = str(PARENT_PHASED_DIR / f'{KID_ID}.bit-vector-sites-mismatches.maternal.bed')

# All CpG sites in the reference genome
OUTPUT_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.parent-phased.all-cpgs'
BED_ALL_CPGS_IN_REFERENCE = str(OUTPUT_DIR / 'all_cpg_sites_in_reference.bed') # output of write_all_cpgs_in_reference.py

# Joint-called VCF
VCF_JOINT_CALLED = str(DEV_DATA_DIR / 'input' / 'CEPH-1463.joint.GRCh38.deepvariant.glnexus.vcf.gz')


## Get all CpG sites in reference genome

In [3]:
import util
importlib.reload(util)
from util import read_all_cpgs_in_reference

DF_ALL_CPGS_IN_REFERENCE = read_all_cpgs_in_reference(BED_ALL_CPGS_IN_REFERENCE)
DF_ALL_CPGS_IN_REFERENCE

chrom,start,end
str,i64,i64
"""chr1""",10468,10469
"""chr1""",10470,10471
"""chr1""",10483,10484
"""chr1""",10488,10489
"""chr1""",10492,10493
"""chr1""",10496,10497
"""chr1""",10524,10525
"""chr1""",10541,10542
"""chr1""",10562,10563


## Read in unphased DNA methylation at CpG sites for kid, dad, and mom

In [4]:
import expand_to_all_cpgs_trio 
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import read_meth_unphased_trio

DF_METH_UNPHASED = read_meth_unphased_trio(
    bed_meth_count_kid=KID_METH_COUNT_COMBINED,
    bed_meth_model_kid=KID_METH_MODEL_COMBINED,
    bed_meth_count_dad=DAD_METH_COUNT_COMBINED,
    bed_meth_model_dad=DAD_METH_MODEL_COMBINED,
    bed_meth_count_mom=MOM_METH_COUNT_COMBINED,
    bed_meth_model_mom=MOM_METH_MODEL_COMBINED,
)
DF_METH_UNPHASED

chrom,start,end,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64
"""chr1""",986903,986904,10,0.6,0.666,11,0.455,0.49,21,0.476,0.538
"""chr1""",986966,986967,10,0.8,0.942,11,0.818,0.897,21,0.524,0.772
"""chr1""",987029,987030,10,0.8,0.941,12,0.667,0.817,21,0.476,0.523
"""chr1""",987116,987117,10,0.5,0.878,13,0.462,0.536,21,0.667,0.842
"""chr1""",987154,987155,11,0.909,0.944,13,0.769,0.864,21,0.619,0.9
"""chr1""",987343,987344,11,0.545,0.843,13,0.846,0.888,22,0.591,0.923
"""chr1""",987366,987367,11,0.909,0.945,13,0.538,0.526,22,0.818,0.946
"""chr1""",987425,987426,12,0.75,0.928,13,0.846,0.923,21,0.905,0.962
"""chr1""",987427,987428,12,0.583,0.91,13,0.846,0.935,22,0.864,0.933


## Read in parent-phased DNA methylation at CpG sites

In [5]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import read_meth_parent_phased

DF_METH_PARENT_PHASED = read_meth_parent_phased(BED_METH_PARENT_PHASED)
DF_METH_PARENT_PHASED

chrom,start,end,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom
str,i64,i64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64
"""chr1""",990862,990863,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990869,990870,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990914,990915,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990924,990925,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990948,990949,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990956,990957,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",991280,991281,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",991431,991432,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",991460,991461,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


## Expand the dataframe of parent-phased methylation levels to include all CpG sites in reference and sample genome, and unphased methylation levels (where available)

In [6]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import expand_meth_to_all_cpgs

DF_METH_PARENT_PHASED_ALL_CPGS = expand_meth_to_all_cpgs(DF_ALL_CPGS_IN_REFERENCE, DF_METH_UNPHASED, DF_METH_PARENT_PHASED)
DF_METH_PARENT_PHASED_ALL_CPGS

chrom,start,end,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64
"""chr1""",10468,10469,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10470,10471,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10483,10484,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10488,10489,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10492,10493,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10496,10497,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10524,10525,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10541,10542,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10562,10563,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


## Add proximity of each CpG site to heterozygous sites at which bit-vectors are mismatched

In [7]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import compute_proximity_to_mismatched_heterozygous_sites

DF_METH_PARENT_PHASED_ALL_CPGS = compute_proximity_to_mismatched_heterozygous_sites(
    DF_METH_PARENT_PHASED_ALL_CPGS,
    BED_HET_SITE_MISMATCHES_PAT,
    BED_HET_SITE_MISMATCHES_MAT,
)
DF_METH_PARENT_PHASED_ALL_CPGS

chrom,start,end,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,is_within_50bp_of_mismatch_site_pat,is_within_50bp_of_mismatch_site_mat
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool
"""chr1""",10468,10469,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10470,10471,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10483,10484,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10488,10489,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10492,10493,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10496,10497,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10524,10525,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10541,10542,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10562,10563,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false


## Bed file enables downstream computation of methylation change upon transmission between generations

https://github.com/quinlan-lab/tapestry/blob/main/images/tapestry.trio.alignments.png

https://github.com/quinlan-lab/tapestry/blob/main/images/tapestry.trio.methylation.png

In [8]:
version_sort(DF_METH_PARENT_PHASED_ALL_CPGS.select([
    "chrom", "start", "end",
    "methylation_level_kid_pat_count", "methylation_level_dad_A_count", "methylation_level_dad_B_count",
    "paternal_haplotype", "paternal_concordance", "is_within_50bp_of_mismatch_site_pat"
]).filter(
    (pl.col('start') > 1989000) & 
    (pl.col('end') < 1989300)
))

chrom,start,end,methylation_level_kid_pat_count,methylation_level_dad_A_count,methylation_level_dad_B_count,paternal_haplotype,paternal_concordance,is_within_50bp_of_mismatch_site_pat
str,i64,i64,f64,f64,f64,str,f64,bool
"""chr1""",1989147,1989148,0.964,null,0.679,"""B""",0.999055,false
"""chr1""",1989153,1989154,0.75,null,0.714,"""B""",0.999055,false
"""chr1""",1989187,1989188,0.821,null,0.929,"""B""",0.999055,false
"""chr1""",1989199,1989200,0.714,null,0.714,"""B""",0.999055,false
"""chr1""",1989206,1989207,0.714,null,0.714,"""B""",0.999055,false
"""chr1""",1989227,1989228,0.679,null,0.571,"""B""",0.999055,false
"""chr1""",1989285,1989286,0.964,null,0.929,"""B""",0.999055,false
"""chr1""",1989291,1989292,0.964,null,0.929,"""B""",0.999055,false


![chr1_1989000_1989300](images/chr1_1989000_1989300.png)

## QC Statistics 

In [9]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import compute_fraction_of_cpgs_that_are_close_to_mismatches

compute_fraction_of_cpgs_that_are_close_to_mismatches(DF_METH_PARENT_PHASED_ALL_CPGS)

Percentage of CpG sites (in reference and sample genome, and on phasable chroms) that are within 50bp of a paternal heterozygous mismatch site: 0.000%
Percentage of CpG sites (in reference and sample genome, and on phasable chroms) that are within 50bp of a maternal heterozygous mismatch site: 0.038%


In [10]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import compute_methylation_coverage_qc

compute_methylation_coverage_qc(DF_METH_PARENT_PHASED_ALL_CPGS)

Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based methylation is phased to kid_pat haplotype: 1.41%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based methylation is phased to kid_mat haplotype: 1.38%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based methylation is phased to at least one parental haplotype (kid): 1.45%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based methylation is phased to both parental haplotypes (kid): 1.33%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based unphased methylation is reported for kid: 1.59%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which model-based methylation is phased to kid_pat haplotype: 1.41%
Percentage of CpG sites (in reference and sample ge

## Overlap CpGs with joint-called SNVs 

In [11]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import get_joint_called_variants

pl.Config.set_tbl_rows(30)

DF_JOINT_CALLED_VARIANTS = get_joint_called_variants(KID_ID, DAD_ID, MOM_ID, VCF_JOINT_CALLED)
DF_JOINT_CALLED_VARIANTS

chrom,start,end,allele_1_kid,allele_2_kid,allele_1_dad,allele_2_dad,allele_1_mom,allele_2_mom
str,i64,i64,str,str,str,str,str,str
"""chr1""",1000078,1000079,"""1""","""1""","""1""","""1""","""1""","""1"""
"""chr1""",1000111,1000112,"""1""","""1""","""1""","""1""","""1""","""1"""
"""chr1""",1000290,1000291,"""0""","""0""","""0""","""0""","""0""","""0"""
"""chr1""",1000334,1000335,"""1""","""0""","""1""","""0""","""1""","""0"""
"""chr1""",1000452,1000453,"""1""","""1""","""1""","""1""","""1""","""1"""
"""chr1""",1000551,1000552,"""0""","""0""","""0""","""0""","""0""","""0"""
"""chr1""",1000730,1000731,"""0""","""1""","""0""","""1""","""0""","""1"""
"""chr1""",1000813,1000814,"""1""","""1""","""1""","""1""","""1""","""1"""
"""chr1""",1000829,1000830,"""1""","""1""","""1""","""1""","""1""","""1"""


In [12]:
# An example of CpG site creation 
# A site that is CpG in only one haplotype of the sample, and not CpG in the reference sequence
# chr1:1173872 A>G creates a CpG on one haplotype, with a methylation level of 0.0 reported on the other haplotype

![chr1-1,173,852-1,173,891](images/chr1-1,173,852-1,173,891.png)


In [13]:
pl.Config.set_tbl_rows(10)

# variant at this CpG site: 
# note that the IGV screenshot (above) reports the pedmed-phased variant, but the dataframe (below) reports the corrresponding unphased variant
DF_JOINT_CALLED_VARIANTS.filter(
    (pl.col("chrom") == 'chr1') & 
    (pl.col("start") == 1173871)
)

chrom,start,end,allele_1_kid,allele_2_kid,allele_1_dad,allele_2_dad,allele_1_mom,allele_2_mom
str,i64,i64,str,str,str,str,str,str
"""chr1""",1173871,1173872,"""0""","""0""","""0""","""1""","""1""","""0"""


In [14]:
# Methylation at this CpG site on Dad B is 0.0
# It should be None as there is no CpG on that haplotype
# Similar for Mom C

DF_METH_PARENT_PHASED_ALL_CPGS.filter(
    (pl.col("chrom") == 'chr1') & 
    (pl.col("start") == 1173870)
)

chrom,start,end,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,is_within_50bp_of_mismatch_site_pat,is_within_50bp_of_mismatch_site_mat
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool
"""chr1""",1173870,1173871,null,null,null,29,0.069,0.024,36,0.056,0.026,null,null,null,null,null,null,13,0.154,16,0.0,0.021,0.029,24,0.0,12,0.167,0.032,0.032,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true


In [15]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import label_with_variants

DF_METH_PARENT_PHASED_ALL_CPGS_WITH_VARIANT_LABEL = label_with_variants(DF_METH_PARENT_PHASED_ALL_CPGS, DF_JOINT_CALLED_VARIANTS)

In [16]:
DF_METH_PARENT_PHASED_ALL_CPGS_WITH_VARIANT_LABEL.filter(
    (pl.col("chrom") == 'chr1') & 
    (pl.col("start_cpg") == 1173870)
)

chrom,start_cpg,end_cpg,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,cpg_is_within_50bp_of_mismatch_site_pat,cpg_is_within_50bp_of_mismatch_site_mat,start_variant,end_variant,allele_1_kid,allele_2_kid,allele_1_dad,allele_2_dad,allele_1_mom,allele_2_mom,num_SNVs_overlapping_CG
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool,i64,i64,str,str,str,str,str,str,u32
"""chr1""",1173870,1173872,null,null,null,29,0.069,0.024,36,0.056,0.026,null,null,null,null,null,null,13,0.154,16,0.0,0.021,0.029,24,0.0,12,0.167,0.032,0.032,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,1173871,1173872,"""0""","""0""","""0""","""1""","""1""","""0""",1


## Label each unique CpG record with a flag indicating whether it is allele-specific

In [17]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import label_cpgs_as_allele_specific

DF_METH_PARENT_PHASED_ALL_CPGS_WITH_ALLELE_SPECIFIC_FLAG = label_cpgs_as_allele_specific(
    DF_METH_PARENT_PHASED_ALL_CPGS_WITH_VARIANT_LABEL,
) 
DF_METH_PARENT_PHASED_ALL_CPGS_WITH_ALLELE_SPECIFIC_FLAG.filter(
    pl.col('cpg_overlaps_at_least_one_snv')
)

chrom,start_cpg,end_cpg,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,cpg_is_within_50bp_of_mismatch_site_pat,cpg_is_within_50bp_of_mismatch_site_mat,cpg_overlaps_at_least_one_snv,snv_genotypes_kid,cpg_is_allele_specific_kid,snv_genotypes_dad,cpg_is_allele_specific_dad,snv_genotypes_mom,cpg_is_allele_specific_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool,bool,str,bool,str,bool,str,bool
"""chr1""",1000110,1000112,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,true,"""hom""",false,"""hom""",false,"""hom""",false
"""chr1""",1000334,1000336,44,0.045,0.036,34,0.029,0.028,49,0.0,0.032,26,0.0,18,0.111,0.025,0.04,13,0.077,21,0.0,0.04,0.02,19,0.0,29,0.0,0.032,0.027,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1000451,1000453,42,0.19,0.037,35,0.114,0.036,45,0.178,0.038,25,0.12,17,0.294,0.03,0.111,13,0.077,22,0.136,0.042,0.041,17,0.235,27,0.148,0.119,0.034,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom""",false,"""hom""",false,"""hom""",false
"""chr1""",1000452,1000454,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,true,"""hom""",false,"""hom""",false,"""hom""",false
"""chr1""",1000551,1000553,44,0.159,0.031,35,0.143,0.064,49,0.082,0.028,26,0.115,18,0.222,0.03,0.075,13,0.077,22,0.182,0.054,0.07,19,0.105,29,0.069,0.027,0.028,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom""",false,"""hom""",false,"""hom""",false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr1""",1998653,1998655,54,0.333,0.399,null,null,null,31,0.548,0.613,23,0.0,null,null,0.056,null,null,null,null,null,null,null,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,null,null,null,null,null,false,false,true,"""het""",true,"""hom""",false,"""hom""",false
"""chr1""",1999113,1999115,null,null,null,28,0.25,0.42,null,null,null,null,null,null,null,null,null,11,0.636,17,0.0,0.712,0.058,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,null,null,null,null,null,false,false,true,"""hom""",false,"""het""",true,"""hom""",false
"""chr1""",1999364,1999366,54,0.444,0.505,29,0.31,0.424,30,0.267,0.195,24,0.0,null,null,0.052,null,12,0.75,17,0.0,0.841,0.062,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,null,null,null,null,null,false,false,true,"""het""",true,"""het""",true,"""hom""",false


### Sanity checking 

In [18]:
# CGs that overlap 1 SNV that is het in kid indeed have zero methylation on one haplotype in kid, and can be flagged for exclusion in imprinting scans in the kid: 
DF_METH_PARENT_PHASED_ALL_CPGS_WITH_ALLELE_SPECIFIC_FLAG.filter(pl.col("snv_genotypes_kid") == "het").sample(10, seed=42)

chrom,start_cpg,end_cpg,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,cpg_is_within_50bp_of_mismatch_site_pat,cpg_is_within_50bp_of_mismatch_site_mat,cpg_overlaps_at_least_one_snv,snv_genotypes_kid,cpg_is_allele_specific_kid,snv_genotypes_dad,cpg_is_allele_specific_dad,snv_genotypes_mom,cpg_is_allele_specific_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool,bool,str,bool,str,bool,str,bool
"""chr1""",1976989,1976991,46,0.196,0.344,34,0.294,0.101,null,null,null,21,0.429,25,0.0,0.564,0.044,null,null,29,0.31,null,0.116,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""hom""",false,"""hom""",false
"""chr1""",1302447,1302449,43,0.442,0.505,22,0.773,0.927,30,0.367,0.504,27,0.704,15,0.0,0.917,0.06,10,0.7,null,null,0.88,null,17,0.0,13,0.846,0.051,0.96,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,true,"""het""",true,"""hom""",false,"""het""",true
"""chr1""",1993897,1993899,50,0.34,0.505,29,0.586,0.576,29,0.448,0.482,22,0.773,28,0.0,0.956,0.053,null,null,21,0.81,null,0.928,17,0.765,12,0.0,0.877,0.059,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1743467,1743469,56,0.304,0.373,24,0.458,0.501,null,null,null,27,0.63,29,0.0,0.65,0.055,null,null,17,0.647,null,0.68,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""hom""",false
"""chr1""",1976572,1976574,46,0.326,0.481,33,0.727,0.851,null,null,null,20,0.75,26,0.0,0.919,0.073,null,null,28,0.679,null,0.787,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""hom""",false,"""hom""",false
"""chr1""",1687151,1687153,41,0.585,0.601,32,0.531,0.53,38,0.474,0.535,17,0.0,24,1.0,0.051,0.951,18,0.944,14,0.0,0.962,0.058,19,0.0,19,0.947,0.056,0.96,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1053550,1053552,34,0.529,0.556,27,0.333,0.597,null,null,null,19,0.947,15,0.0,0.957,0.062,17,0.0,10,0.9,0.082,0.955,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""hom""",false
"""chr1""",1688585,1688587,43,0.395,0.456,30,0.367,0.482,40,0.375,0.492,20,0.85,23,0.0,0.941,0.054,18,0.0,12,0.917,0.05,0.968,18,0.833,22,0.0,0.958,0.047,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1077570,1077572,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,true

In [19]:
# CGs that overlap 1 SNV that is het in dad indeed have zero methylation on one haplotype in dad, and can be flagged for exclusion in imprinting scans in dad: 
DF_METH_PARENT_PHASED_ALL_CPGS_WITH_ALLELE_SPECIFIC_FLAG.filter(pl.col("snv_genotypes_dad") == "het").sample(10, seed=42)

chrom,start_cpg,end_cpg,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,cpg_is_within_50bp_of_mismatch_site_pat,cpg_is_within_50bp_of_mismatch_site_mat,cpg_overlaps_at_least_one_snv,snv_genotypes_kid,cpg_is_allele_specific_kid,snv_genotypes_dad,cpg_is_allele_specific_dad,snv_genotypes_mom,cpg_is_allele_specific_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool,bool,str,bool,str,bool,str,bool
"""chr1""",1792925,1792927,80,0.75,0.957,32,0.5,0.533,54,0.333,0.521,40,0.75,24,0.625,0.96,0.923,11,0.0,21,0.762,0.063,0.935,29,0.0,25,0.72,0.061,0.962,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom""",false,"""het""",true,"""het""",true
"""chr1""",1453960,1453962,40,0.65,0.703,22,0.182,0.096,24,0.333,0.124,21,0.571,18,0.722,0.584,0.786,10,0.0,12,0.333,0.048,0.312,null,null,10,0.3,null,0.331,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom""",false,"""het""",true,"""hom""",false
"""chr1""",1981542,1981544,45,0.467,0.55,null,null,null,30,0.9,0.963,22,0.0,23,0.913,0.063,0.937,null,null,null,null,null,null,13,0.923,14,0.857,0.967,0.952,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,false,""".""",false,"""het""",true,"""hom""",false
"""chr1""",1734262,1734264,null,null,null,26,0.269,0.496,44,0.25,0.312,null,null,null,null,null,null,10,0.7,16,0.0,0.931,0.063,21,0.524,23,0.0,0.563,0.058,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom""",false,"""het""",true,"""het""",true
"""chr1""",1773847,1773849,null,null,null,52,0.385,0.519,44,0.386,0.532,null,null,null,null,null,null,24,0.833,28,0.0,0.963,0.057,17,1.0,27,0.0,0.963,0.062,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom""",false,"""het""",true,"""het""",true
"""chr1""",1671993,1671995,53,0.396,0.478,36,0.444,0.511,39,0.359,0.417,24,0.0,29,0.724,0.054,0.892,17,0.941,19,0.0,0.947,0.06,20,0.0,19,0.737,0.048,0.861,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1072050,1072052,24,0.375,0.442,17,0.176,0.384,16,0.188,0.287,null,null,15,0.0,null,0.04,10,0.0,null,null,0.047,null,12,0.0,null,null,0.04,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1684265,1684267,38,0.447,0.52,29,0.483,0.542,40,0.4,0.543,18,0.0,19,0.842,0.055,0.945,17,0.824,12,0.0,0.964,0.062,21,0.0,19,0.842,0.06,0.963,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1163333,1163335,null,null,null,31,0.387,0.433,null,null,null,null,null,null,null,null,null,17,0.706,14,0.0,0.886,0.065,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,fa

In [20]:
# CGs that overlap 1 SNV that is het in mom indeed have zero methylation on one haplotype in mom, and can be flagged for exclusion in imprinting scans in mom: 
DF_METH_PARENT_PHASED_ALL_CPGS_WITH_ALLELE_SPECIFIC_FLAG.filter(pl.col("snv_genotypes_mom") == "het").sample(10, seed=42)

chrom,start_cpg,end_cpg,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,cpg_is_within_50bp_of_mismatch_site_pat,cpg_is_within_50bp_of_mismatch_site_mat,cpg_overlaps_at_least_one_snv,snv_genotypes_kid,cpg_is_allele_specific_kid,snv_genotypes_dad,cpg_is_allele_specific_dad,snv_genotypes_mom,cpg_is_allele_specific_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool,bool,str,bool,str,bool,str,bool
"""chr1""",1837454,1837456,null,null,null,36,0.306,0.496,44,0.5,0.543,null,null,null,null,null,null,15,0.733,21,0.0,0.961,0.057,23,0.957,21,0.0,0.962,0.058,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom""",false,"""het""",true,"""het""",true
"""chr1""",1437954,1437956,26,0.577,0.559,19,0.474,0.508,12,0.5,0.538,16,0.938,10,0.0,0.916,0.052,null,null,11,0.818,null,0.872,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1950070,1950072,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,true,"""hom""",false,"""het""",true,"""het""",true
"""chr1""",1742468,1742470,null,null,null,null,null,null,44,0.455,0.552,null,null,null,null,null,null,null,null,null,null,null,null,23,0.87,21,0.0,0.964,0.06,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom""",false,"""hom""",false,"""het""",true
"""chr1""",1795440,1795442,67,0.597,0.575,32,0.844,0.947,53,0.491,0.532,42,0.952,25,0.0,0.966,0.045,13,0.846,19,0.842,0.948,0.946,29,0.897,24,0.0,0.958,0.055,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""hom""",false,"""het""",true
"""chr1""",1699126,1699128,53,0.358,0.547,34,0.235,0.457,40,0.25,0.494,25,0.76,28,0.0,0.94,0.057,18,0.0,16,0.5,0.053,0.895,19,0.526,21,0.0,0.957,0.042,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1065262,1065264,23,0.217,0.424,14,0.357,0.526,21,0.333,0.524,null,null,14,0.0,null,0.04,null,null,null,null,null,null,13,0.0,null,null,0.058,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1705854,1705856,48,0.417,0.551,28,0.429,0.617,38,0.368,0.496,24,0.0,24,0.833,0.05,0.914,16,0.75,12,0.0,0.94,0.06,20,0.0,18,0.778,0.053,0.821,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het""",true,"""het""",true,"""het""",true
"""chr1""",1127257,1127259,51,0.471,0.556,23,0.522,0.514,37,0.351,0.47,28,0.857,23,0.0,0.953,0.05,null,null,16,0.75,null,0.909,20,0.0,17,0.765,0.049,0.83,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,true,"""h

In [21]:
# CGs that overlap 2 SNVs, the first of which is het in kid, indeed have zero methylation on one haplotype in kid, or have null methylation levels because of a deletion, and can be flagged for exclusion in imprinting scans in kid: 
DF_METH_PARENT_PHASED_ALL_CPGS_WITH_ALLELE_SPECIFIC_FLAG.filter(pl.col("snv_genotypes_kid").str.contains("het,")) 

chrom,start_cpg,end_cpg,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,cpg_is_within_50bp_of_mismatch_site_pat,cpg_is_within_50bp_of_mismatch_site_mat,cpg_overlaps_at_least_one_snv,snv_genotypes_kid,cpg_is_allele_specific_kid,snv_genotypes_dad,cpg_is_allele_specific_dad,snv_genotypes_mom,cpg_is_allele_specific_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool,bool,str,bool,str,bool,str,bool
"""chr1""",1076849,1076851,11,0.545,0.706,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,true,"""het,hom""",true,""".,.""",false,"""hom,hom""",false
"""chr1""",1077239,1077241,10,0.6,0.553,null,null,null,null,null,null,10,0.6,null,null,0.553,null,null,null,null,null,null,null,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het,hom""",true,""".,.""",false,"""hom,hom""",false
"""chr1""",1077808,1077810,19,0.316,0.48,11,0.091,0.113,null,null,null,12,0.5,null,null,0.58,null,null,null,null,null,null,null,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het,hom""",true,""".,.""",false,"""het,hom""",true
"""chr1""",1079339,1079341,27,0.0,0.032,null,null,null,null,null,null,14,0.0,13,0.0,0.039,0.029,null,null,null,null,null,null,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,true,"""het,het""",true,"""het,het""",true,"""het,het""",true
"""chr1""",1606010,1606012,null,null,null,null,null,null,32,0.375,0.619,null,null,null,null,null,null,null,null,null,null,null,null,16,0.75,16,0.0,0.857,0.063,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het,het""",true,""".,hom""",false,"""het,hom""",true


In [22]:
# CGs that overlap 2 SNVs, the second of which is het in kid, indeed have zero methylation on one haplotype in kid, or have null methylation levels because of a deletion, and can be flagged for exclusion in imprinting scans in kid: 
DF_METH_PARENT_PHASED_ALL_CPGS_WITH_ALLELE_SPECIFIC_FLAG.filter(pl.col("snv_genotypes_kid").str.contains(",het"))

chrom,start_cpg,end_cpg,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,cpg_is_within_50bp_of_mismatch_site_pat,cpg_is_within_50bp_of_mismatch_site_mat,cpg_overlaps_at_least_one_snv,snv_genotypes_kid,cpg_is_allele_specific_kid,snv_genotypes_dad,cpg_is_allele_specific_dad,snv_genotypes_mom,cpg_is_allele_specific_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool,bool,str,bool,str,bool,str,bool
"""chr1""",1079339,1079341,27,0.0,0.032,null,null,null,null,null,null,14,0.0,13,0.0,0.039,0.029,null,null,null,null,null,null,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,true,"""het,het""",true,"""het,het""",true,"""het,het""",true
"""chr1""",1606010,1606012,null,null,null,null,null,null,32,0.375,0.619,null,null,null,null,null,null,null,null,null,null,null,null,16,0.75,16,0.0,0.857,0.063,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""het,het""",true,""".,hom""",false,"""het,hom""",true


In [23]:
# CGs that overlap 2 SNVs, the first of which is het in dad, indeed have zero methylation on one haplotype in dad, or have null methylation levels because of a deletion, and can be flagged for exclusion in imprinting scans in dad: 
DF_METH_PARENT_PHASED_ALL_CPGS_WITH_ALLELE_SPECIFIC_FLAG.filter(pl.col("snv_genotypes_dad").str.contains("het,"))

chrom,start_cpg,end_cpg,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,cpg_is_within_50bp_of_mismatch_site_pat,cpg_is_within_50bp_of_mismatch_site_mat,cpg_overlaps_at_least_one_snv,snv_genotypes_kid,cpg_is_allele_specific_kid,snv_genotypes_dad,cpg_is_allele_specific_dad,snv_genotypes_mom,cpg_is_allele_specific_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool,bool,str,bool,str,bool,str,bool
"""chr1""",1079339,1079341,27,0.0,0.032,null,null,null,null,null,null,14,0.0,13,0.0,0.039,0.029,null,null,null,null,null,null,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,true,"""het,het""",true,"""het,het""",true,"""het,het""",true
"""chr1""",1497604,1497606,null,null,null,36,0.444,0.619,null,null,null,null,null,null,null,null,null,18,0.889,18,0.0,0.934,0.06,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom,hom""",false,"""het,het""",true,"""hom,hom""",false
"""chr1""",1534217,1534219,57,0.947,0.947,28,0.429,0.5,42,0.881,0.946,28,0.893,29,1.0,0.938,0.949,13,0.0,15,0.8,0.048,0.82,22,0.909,20,0.85,0.952,0.941,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom,hom""",false,"""het,het""",true,"""hom,hom""",false


In [24]:
# CGs that overlap 2 SNVs, the second of which is het in dad, indeed have zero methylation on one haplotype in dad, or have null methylation levels because of a deletion, and can be flagged for exclusion in imprinting scans in dad: 
DF_METH_PARENT_PHASED_ALL_CPGS_WITH_ALLELE_SPECIFIC_FLAG.filter(pl.col("snv_genotypes_dad").str.contains(",het"))

chrom,start_cpg,end_cpg,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,cpg_is_within_50bp_of_mismatch_site_pat,cpg_is_within_50bp_of_mismatch_site_mat,cpg_overlaps_at_least_one_snv,snv_genotypes_kid,cpg_is_allele_specific_kid,snv_genotypes_dad,cpg_is_allele_specific_dad,snv_genotypes_mom,cpg_is_allele_specific_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool,bool,str,bool,str,bool,str,bool
"""chr1""",1079339,1079341,27,0.0,0.032,null,null,null,null,null,null,14,0.0,13,0.0,0.039,0.029,null,null,null,null,null,null,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,true,true,"""het,het""",true,"""het,het""",true,"""het,het""",true
"""chr1""",1497604,1497606,null,null,null,36,0.444,0.619,null,null,null,null,null,null,null,null,null,18,0.889,18,0.0,0.934,0.06,null,null,null,null,null,null,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom,hom""",false,"""het,het""",true,"""hom,hom""",false
"""chr1""",1534217,1534219,57,0.947,0.947,28,0.429,0.5,42,0.881,0.946,28,0.893,29,1.0,0.938,0.949,13,0.0,15,0.8,0.048,0.82,22,0.909,20,0.85,0.952,0.941,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,"""hom,hom""",false,"""het,het""",true,"""hom,hom""",false
"""chr1""",1605667,1605669,48,0.417,0.5,null,null,null,32,0.375,0.643,18,0.0,30,0.667,0.044,0.5,null,null,null,null,null,null,16,0.0,16,0.75,0.046,0.708,1000334,1999756,"""B""",0.999055,1058,1000334,1993898,"""D""",0.721598,801,false,false,true,""".,.""",false,""".,het""",true,"""het,het""",true
"""chr1""",1924206,1924208,null,null,null,11,0.455,0.472,13,0.385,0.451,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,true,"""hom,hom""",false,"""hom,het""",true,"""hom,het""",true
